In [3]:
# -*- coding: utf-8 -*-
import matplotlib.pyplot as plt
import numpy as np
import torch
import h5py
from tqdm import tqdm
from monai.inferers import SlidingWindowInferer

from model.SCRN import SCRN

In [30]:
def plot_result(clean_data, sparse_data, recon_data, diff_data, save_path, title):
    """
    Отрисовка в ИСХОДНОМ масштабе амплитуд (без нормализации)
    """
    vmax = np.max(np.abs(clean_data))

    fig, axes = plt.subplots(1, 4, figsize=(18, 6))

    axes[0].imshow(clean_data, aspect="auto", cmap="Grays", origin="upper", vmin=-vmax, vmax=vmax)
    axes[0].set_title("Исходные данные")
    axes[0].set_xlabel("Трассы")
    axes[0].set_ylabel("Отсчеты по времени")

    axes[1].imshow(sparse_data, aspect="auto", cmap="Grays", origin="upper", vmin=-vmax, vmax=vmax)
    axes[1].set_title("Прореженные данные")
    axes[1].set_xlabel("Трассы")

    axes[2].imshow(recon_data, aspect="auto", cmap="Grays", origin="upper", vmin=-vmax, vmax=vmax)
    axes[2].set_title("Результат работы модели")
    axes[2].set_xlabel("Трассы")

    axes[3].imshow(diff_data, aspect="auto", cmap="Grays", origin="upper", vmin=-vmax, vmax=vmax)
    axes[3].set_title("Разница")
    axes[3].set_xlabel("Трассы")

    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

    np.save('results/npy_files/second_clean_data.npy', clean_data)
    np.save('results/npy_files/second_sparse_data.npy', sparse_data)
    np.save('results/npy_files/second_recon_data.npy', recon_data)

In [5]:
def stitch_patches(patches, original_shape, patch_size=128, stride=1.0):
    """
    Должны подаваться патчи (128, 128).
    """
    h, w = original_shape
    step = int(patch_size * stride)

    out = np.zeros(original_shape, dtype=np.float32)
    count = np.zeros(original_shape, dtype=np.float32)

    i = 0
    for y in range(0, h - patch_size + 1, step):
        for x in range(0, w - patch_size + 1, step):
            patch = patches[i]

            out[y:y + patch_size, x:x + patch_size] += patch
            count[y:y + patch_size, x:x + patch_size] += 1
            i += 1

    mask = count > 0
    out[mask] /= count[mask]
    return out, mask

In [6]:
MODEL_PATH = 'trained_model/best_train_model_second_data.pth'
DATA_PATH = 'inputs/data/test-4-128.hdf5'

device = 'cuda:2' if torch.cuda.is_available() else 'cpu'

In [7]:
model = SCRN()
state_dict = torch.load(MODEL_PATH)
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print()

Код для отрисовки графиков:

In [31]:
with h5py.File(DATA_PATH, 'r') as f:
    image = f['image'][12][0].astype(np.float32)
    label = f['label'][12][0].astype(np.float32)

    input_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).float()
    input_tensor = input_tensor.to(device)
    
    with torch.no_grad():
        predict = model(input_tensor)

    predict = predict.squeeze().cpu().numpy()

    diff_data = np.abs(predict - label)
    plot_result(label, image, predict, diff_data, 'result_predict.png', 'Результат работы нейросети')

Код для склеивания сейсмограмм:

In [32]:
with h5py.File(DATA_PATH, 'r') as f:
    # f_len = len(f['image'])
    predict_list = []

    tqdm_range = tqdm(range(12), leave=True)
    for i in tqdm_range:
        image = f['image'][i][0].astype(np.float32)

        input_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).float()
        input_tensor = input_tensor.to(device)
        
        with torch.no_grad():
            predict = model(input_tensor)

        predict = predict.squeeze().cpu().numpy()
        predict_list.append(predict)

        if i != 0 and (i + 1) % 12 == 0:
            reconstructed, _ = stitch_patches(patches=predict_list, original_shape=(1600, 156), patch_size=128, stride=1.0)

            plt.figure(figsize=(15, 8))
            plt.imshow(reconstructed, cmap='Grays', aspect='auto', origin='upper')
            plt.title('Восстановленная сейсмограмма')
            plt.xlabel('Трассы')
            plt.ylabel('Отсчеты по времени')
            plt.tight_layout()
            plt.savefig(f'results/seimogram/recon_seis_{i // 12 + 1}.png', dpi=150, bbox_inches='tight')
            plt.close()

            np.save(f'results/npy_files/second_stitch_seis_data_{i // 12 + 1}.npy', predict_list)

            predict_list = []

100%|██████████| 12/12 [00:01<00:00,  9.98it/s]


Код для склеивания сейсмограмм через инферер:

In [15]:
inferer = SlidingWindowInferer(
    roi_size=(128, 128),
    sw_batch_size=1,
    overlap=0.8,
    mode="gaussian",
    sigma_scale=0.125,
    padding_mode="constant",
    cval=0.0,
    device=device,
    # progress=True
)

In [ ]:
with h5py.File(DATA_PATH, 'r') as f:
    # f_len = len(f['image'])
    reconstructed = []

    tqdm_range = tqdm(range(12), leave=True)
    for i in tqdm_range:
        image = f['image'][i][0].astype(np.float32)
        label = f['label'][i][0].astype(np.float32)

        input_tensor = torch.from_numpy(image).unsqueeze(0).unsqueeze(0).float()
        input_tensor = input_tensor.to(device)
        
        with torch.no_grad():
            predict = inferer(input_tensor, model)
        
        predict = predict.squeeze().cpu().numpy()
        reconstructed.append(predict)

    reconstructed, _ = stitch_patches(patches=reconstructed, original_shape=(1600, 156), patch_size=128, stride=1.0)
    
    plt.figure(figsize=(15, 10))
    plt.imshow(reconstructed, cmap='Grays', aspect='auto', origin='upper')
    plt.title('Восстановленная сейсмограмма')
    plt.xlabel('Трассы')
    plt.ylabel('Отсчеты по времени')
    plt.tight_layout()
    plt.savefig('recon_seis_inferer.png', dpi=150, bbox_inches='tight')
    plt.close()


100%|██████████| 12/12 [00:00<00:00, 51.02it/s]
